# Local Training — Surface Crack Detection

**Auto-detect hardware, train 3 models, track sessions, compare results.**

In [ ]:
# Optional: detect existing checkpoints and session history
check = input("Check for existing model checkpoints? (Y/n): ").strip().lower()
if check not in ("n", "no"):
    from src.session import SessionTracker
    past = SessionTracker.past_models()
    if past:
        print("Existing checkpoints:")
        for name, info in past.items():
            print(f"  {name:20s} {info['size_mb']:6.1f} MB  ({info['modified']})")
    summary = SessionTracker.summary()
    if "No past sessions" not in summary:
        print(f"\n{summary}")
else:
    print("Skipped model detection.")
print("Model detection complete.")


In [ ]:
import sys, os, json, time as _time, traceback

def _find_root():
    root = os.getcwd()
    for _ in range(10):
        if os.path.isfile(os.path.join(root, "src", "config.py")):
            return root
        parent = os.path.dirname(root)
        if parent == root:
            break
        root = parent
    print("ERROR: Cannot find project root (no src/config.py)")
    return None

PROJECT_DIR = _find_root()
if PROJECT_DIR:
    os.chdir(PROJECT_DIR)
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)
    print(f"Project root: {PROJECT_DIR}")

In [ ]:
import sys, os; PROJECT_DIR = os.getcwd() if os.path.isfile(os.path.join(os.getcwd(), "src", "config.py")) else next((d for d in [os.getcwd(), os.path.join(os.getcwd(), "..")] if os.path.isfile(os.path.join(d, "src", "config.py"))), None); _ = [sys.path.insert(0, p) for p in ([PROJECT_DIR] if PROJECT_DIR and PROJECT_DIR not in sys.path else [])]
from src.monitor import detect_hardware, auto_monitor
import torch
import src.config as cfg

detect_hardware(verbose=True)

devices = {}
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    devices["cuda"] = f"GPU {torch.cuda.get_device_name(0)} ({props.total_memory / 1024**3:.1f} GB)"
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    devices["mps"] = "Apple MPS (Metal)"
if hasattr(torch, "xpu") and torch.xpu.is_available():
    devices["xpu"] = "Intel XPU"
devices["cpu"] = "CPU"

keys = list(devices.keys())
print("\nDetected compute devices:")
for i, k in enumerate(keys):
    print(f"  [{i}] {k.upper()}: {devices[k]}")

non_cpu = [k for k in keys if k != "cpu"]
default_idx = 0 if non_cpu else keys.index("cpu")
try:
    choice = input(f"Select device [0-{len(keys)-1}], Enter=auto ({keys[default_idx].upper()}): ").strip()
    selected = keys[int(choice)] if choice else keys[default_idx]
except (ValueError, IndexError):
    print(f"Invalid choice, using default: {keys[default_idx].upper()}")
    selected = keys[default_idx]

cfg.Config.DEVICE = torch.device(selected)
print(f"\nUsing: {selected.upper()} — {devices[selected]}")

auto_monitor()


In [ ]:
import sys, os; PROJECT_DIR = os.getcwd() if os.path.isfile(os.path.join(os.getcwd(), "src", "config.py")) else next((d for d in [os.getcwd(), os.path.join(os.getcwd(), "..")] if os.path.isfile(os.path.join(d, "src", "config.py"))), None); _ = [sys.path.insert(0, p) for p in ([PROJECT_DIR] if PROJECT_DIR and PROJECT_DIR not in sys.path else [])]
from src.session import SessionTracker

past = SessionTracker.past_models()
if past:
    print("Found existing checkpoints:")
    for name, info in past.items():
        print(f"  {name:20s} {info['size_mb']:6.1f} MB  (modified: {info['modified']})")

summary = SessionTracker.summary()
if "No past sessions" not in summary:
    print(f"\n{summary}")

retrain = input("\nRetrain all models? (Y/n): ").strip().lower()
SKIP_TRAINING = retrain in ("n", "no")
if SKIP_TRAINING:
    print("Skipping training — will evaluate existing models.")

In [ ]:
import sys, os; PROJECT_DIR = os.getcwd() if os.path.isfile(os.path.join(os.getcwd(), "src", "config.py")) else next((d for d in [os.getcwd(), os.path.join(os.getcwd(), "..")] if os.path.isfile(os.path.join(d, "src", "config.py"))), None); _ = [sys.path.insert(0, p) for p in ([PROJECT_DIR] if PROJECT_DIR and PROJECT_DIR not in sys.path else [])]
import src.config as cfg

if SKIP_TRAINING:
    QUICK_MODE = False
    print("Skipping training setup.")
else:
    print("Quick test: 1 epoch per model. Full run: 5 + 15 epochs.")
    q = input("Run quick test? (y/N): ").strip().lower()
    QUICK_MODE = q in ("y", "yes")
    if QUICK_MODE:
        cfg.Config.EPOCHS_WARMUP = 1
        cfg.Config.EPOCHS_FINE_TUNE = 1
        cfg.Config.EARLY_STOP_PATIENCE = 5
        print("Quick mode: 1 warmup + 1 fine-tune.")
    else:
        print("Full mode: 5 warmup + 15 fine-tune.")

In [ ]:
import sys, os, zipfile; PROJECT_DIR = os.getcwd() if os.path.isfile(os.path.join(os.getcwd(), "src", "config.py")) else next((d for d in [os.getcwd(), os.path.join(os.getcwd(), "..")] if os.path.isfile(os.path.join(d, "src", "config.py"))), None); _ = [sys.path.insert(0, p) for p in ([PROJECT_DIR] if PROJECT_DIR and PROJECT_DIR not in sys.path else [])]
from huggingface_hub import hf_hub_download
from src.prepare_data import prepare_data
from src.dataset import get_dataloaders

RAW_DIR = "data/raw"

if os.path.exists(RAW_DIR) and os.listdir(RAW_DIR):
    print(f"Dataset found: {RAW_DIR}/")
else:
    print(f"Local dataset not found at {RAW_DIR}/")
    dl = input("Download from HF Hub? (Y/n): ").strip().lower()
    if dl not in ("n", "no"):
        try:
            os.makedirs("data", exist_ok=True)
            zip_path = hf_hub_download(repo_id="amruthjakku/surface-crack-detection-model", filename="dataset.zip", repo_type="model")
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall("data")
            print(f"Extracted to {RAW_DIR}/")
        except Exception as e:
            print(f"Download failed: {e}")
            print("Place images in data/raw/ with subfolders: Cracks/, Patch/, Potholes/, Surface Defects/")
            raise SystemExit(1)
    else:
        print("Place images in data/raw/ with class subfolders and re-run.")
        raise SystemExit(1)

prepare_data()
train_loader, val_loader, test_loader = get_dataloaders()
print(f"Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}, Test: {len(test_loader.dataset)}")
if len(train_loader.dataset) == 0:
    print("ERROR: Training set empty.")
    raise SystemExit(1)

In [ ]:
# Pre-training model sync: download latest checkpoints from HF Hub
import sys, os; PROJECT_DIR = os.getcwd() if os.path.isfile(os.path.join(os.getcwd(), "src", "config.py")) else next((d for d in [os.getcwd(), os.path.join(os.getcwd(), "..")] if os.path.isfile(os.path.join(d, "src", "config.py"))), None); _ = [sys.path.insert(0, p) for p in ([PROJECT_DIR] if PROJECT_DIR and PROJECT_DIR not in sys.path else [])]
import src.config as cfg
from src.utils import sync_models_from_hub

print("--- Pre-training Model Sync ---")
sync_input = input("Sync model checkpoints from HF Hub? (y/N): ").strip().lower()
if sync_input in ("y", "yes"):
    token = input("HF token (or Enter if public repo): ").strip() or None
    n = sync_models_from_hub(token=token)
    print(f"Sync complete: {n} file(s) downloaded.")
else:
    print("Skipped model sync.")


In [ ]:
import sys, os, time as _time, traceback; PROJECT_DIR = os.getcwd() if os.path.isfile(os.path.join(os.getcwd(), "src", "config.py")) else next((d for d in [os.getcwd(), os.path.join(os.getcwd(), "..")] if os.path.isfile(os.path.join(d, "src", "config.py"))), None); _ = [sys.path.insert(0, p) for p in ([PROJECT_DIR] if PROJECT_DIR and PROJECT_DIR not in sys.path else [])]
import torch
import src.config as cfg
from src.train import run_training
from src.evaluate import run_evaluation
from src.session import SessionTracker

MODELS = ["resnet50", "efficientnet_b0", "vit_b_16"]
model_results = {}

for model_name in MODELS:
    if SKIP_TRAINING:
        path = cfg.Config.get_model_path(model_name)
        if os.path.exists(path):
            print(f"\n{'=' * 60}")
            print(f"{model_name} — evaluating existing checkpoint")
            print(f"{'=' * 60}")
            try:
                metrics = run_evaluation(model_name=model_name)
                if metrics:
                    model_results[model_name] = {"status": "ok", **metrics}
                else:
                    model_results[model_name] = {"status": "ok"}
            except Exception as e:
                print(f"Evaluation failed: {e}")
                model_results[model_name] = {"status": "failed", "error": str(e)[:200]}
        else:
            print(f"\nSkipping {model_name} — no checkpoint.")
            model_results[model_name] = {"status": "skipped"}
        continue

    print(f"\n{'=' * 60}")
    print(f"Training {model_name}...")
    print(f"{'=' * 60}")
    t0 = _time.time()
    try:
        train_result = run_training(model_name=model_name)
        elapsed = _time.time() - t0
        print(f"\nTraining time: {elapsed / 60:.1f} min")
        path = cfg.Config.get_model_path(model_name)
        if os.path.exists(path):
            print(f"Checkpoint: {path} ({os.path.getsize(path)/1024/1024:.1f} MB)")
            metrics = run_evaluation(model_name=model_name)
            if metrics:
                model_results[model_name] = {"status": "ok", **metrics}
            else:
                model_results[model_name] = {"status": "ok"}
        else:
            print("No checkpoint saved.")
            model_results[model_name] = {"status": "failed", "error": "No checkpoint"}
    except Exception as e:
        traceback.print_exc()
        print(f"{model_name} failed: {e}")
        model_results[model_name] = {"status": "failed", "error": str(e)[:200]}
    torch.cuda.empty_cache()

print("\nTraining sequence complete.")

In [ ]:
import sys, os; PROJECT_DIR = os.getcwd() if os.path.isfile(os.path.join(os.getcwd(), "src", "config.py")) else next((d for d in [os.getcwd(), os.path.join(os.getcwd(), "..")] if os.path.isfile(os.path.join(d, "src", "config.py"))), None); _ = [sys.path.insert(0, p) for p in ([PROJECT_DIR] if PROJECT_DIR and PROJECT_DIR not in sys.path else [])]
import src.config as cfg
from src.train import run_kfold

if SKIP_TRAINING:
    print("Skipping K-Fold CV (no training was run).")
else:
    print("\n" + "=" * 60)
    print("5-FOLD CROSS-VALIDATION")
    print("=" * 60)
    kfold_result = run_kfold(model_name=cfg.Config.MODEL_NAME)
    print("\nK-Fold CV complete.")


In [ ]:
import sys, os; PROJECT_DIR = os.getcwd() if os.path.isfile(os.path.join(os.getcwd(), "src", "config.py")) else next((d for d in [os.getcwd(), os.path.join(os.getcwd(), "..")] if os.path.isfile(os.path.join(d, "src", "config.py"))), None); _ = [sys.path.insert(0, p) for p in ([PROJECT_DIR] if PROJECT_DIR and PROJECT_DIR not in sys.path else [])]
import torch, numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report
from src.dataset import get_dataloaders
from src.model import get_model
import src.config as cfg

models_to_load = [m for m in cfg.Config.ENSEMBLE_MODELS if os.path.exists(cfg.Config.get_model_path(m))]
ensemble_metrics = {}

if len(models_to_load) < 2:
    print(f"Ensemble skipped — need >= 2 trained models, have {len(models_to_load)}")
else:
    device = cfg.Config.DEVICE
    nets = []
    for name in models_to_load:
        net = get_model(model_name=name, num_classes=cfg.Config.NUM_CLASSES, pretrained=False)
        net.load_state_dict(torch.load(cfg.Config.get_model_path(name), map_location=device))
        nets.append(net.to(device).eval())

    _, _, test_loader = get_dataloaders()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            probs = sum(torch.softmax(net(images), dim=1) for net in nets)
            y_true.extend(labels.numpy())
            y_pred.extend(torch.max(probs, 1)[1].cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    report = classification_report(y_true, y_pred, target_names=cfg.Config.CLASSES, zero_division=0)

    print(f"\n{'=' * 60}")
    print(f"Ensemble ({'+'.join(models_to_load)}) Results")
    print(f"{'=' * 60}")
    print(f"Accuracy:   {acc:.4f}")
    print(f"Weighted F1: {f1:.4f}")
    print(f"\n{report}")
    ensemble_metrics = {"accuracy": round(acc, 4), "weighted_f1": round(f1, 4)}

In [ ]:
import sys, os; PROJECT_DIR = os.getcwd() if os.path.isfile(os.path.join(os.getcwd(), "src", "config.py")) else next((d for d in [os.getcwd(), os.path.join(os.getcwd(), "..")] if os.path.isfile(os.path.join(d, "src", "config.py"))), None); _ = [sys.path.insert(0, p) for p in ([PROJECT_DIR] if PROJECT_DIR and PROJECT_DIR not in sys.path else [])]
import src.config as cfg
from src.distill import run_distillation

print("\n" + "=" * 60)
print("KNOWLEDGE DISTILLATION")
print("=" * 60)
print(f"Student: {cfg.Config.DISTILL_STUDENT}")
print(f"Teachers: {cfg.Config.DISTILL_TEACHERS}")

student_metrics = run_distillation()

student_path = os.path.join(cfg.Config.MODELS_DIR, f"student_{cfg.Config.DISTILL_STUDENT}.pth")
if student_metrics and os.path.exists(student_path):
    size_mb = os.path.getsize(student_path) / (1024 * 1024)
    print(f"\nStudent checkpoint: {student_path} ({size_mb:.1f} MB)")
    print(f"Best val loss: {student_metrics["best_val_loss"]:.4f}")
    print(f"Best val acc: {student_metrics["best_val_acc"]:.4f}")

print("\nDistillation complete.")


In [ ]:
import sys, os; PROJECT_DIR = os.getcwd() if os.path.isfile(os.path.join(os.getcwd(), "src", "config.py")) else next((d for d in [os.getcwd(), os.path.join(os.getcwd(), "..")] if os.path.isfile(os.path.join(d, "src", "config.py"))), None); _ = [sys.path.insert(0, p) for p in ([PROJECT_DIR] if PROJECT_DIR and PROJECT_DIR not in sys.path else [])]
import src.config as cfg

synth_input = input("\nGenerate synthetic data + quick retrain? (y/N): ").strip().lower()
if synth_input in ("y", "yes"):
    print("\n--- Generating Synthetic Images ---")
    print("NOTE: Requires diffusers + transformers + a GPU with 8+ GB VRAM.")
    from src.generate_synthetic import generate_synthetic
    try:
        cfg.Config.SYNTHETIC_ENABLED = True
        generate_synthetic()
        print("\n--- Quick retrain with synthetic data ---")
        from src.train import run_training
        orig_epochs = (cfg.Config.EPOCHS_WARMUP, cfg.Config.EPOCHS_FINE_TUNE)
        cfg.Config.EPOCHS_WARMUP = 1
        cfg.Config.EPOCHS_FINE_TUNE = 2
        run_training(model_name=cfg.Config.MODEL_NAME)
        cfg.Config.EPOCHS_WARMUP, cfg.Config.EPOCHS_FINE_TUNE = orig_epochs
        print("Synthetic augmentation + retrain complete.")
    except Exception as e:
        print(f"Synthetic augmentation failed: {e}")
        cfg.Config.SYNTHETIC_ENABLED = False
else:
    print("Skipped synthetic augmentation.")


In [ ]:
import sys, os; PROJECT_DIR = os.getcwd() if os.path.isfile(os.path.join(os.getcwd(), "src", "config.py")) else next((d for d in [os.getcwd(), os.path.join(os.getcwd(), "..")] if os.path.isfile(os.path.join(d, "src", "config.py"))), None); _ = [sys.path.insert(0, p) for p in ([PROJECT_DIR] if PROJECT_DIR and PROJECT_DIR not in sys.path else [])]
from src.session import SessionTracker

all_models = {**model_results}
if ensemble_metrics:
    all_models["ensemble"] = {"status": "ok", "models_used": cfg.Config.ENSEMBLE_MODELS, **ensemble_metrics}

entry = SessionTracker.new_session(
    device=cfg.Config.DEVICE,
    mode="quick" if QUICK_MODE else "full",
    model_results=all_models,
    kfold_summary=kfold_result if "kfold_result" in dir() and kfold_result else None
)
print(f"Session saved: {entry['timestamp']}")

In [ ]:
import sys, os; PROJECT_DIR = os.getcwd() if os.path.isfile(os.path.join(os.getcwd(), "src", "config.py")) else next((d for d in [os.getcwd(), os.path.join(os.getcwd(), "..")] if os.path.isfile(os.path.join(d, "src", "config.py"))), None); _ = [sys.path.insert(0, p) for p in ([PROJECT_DIR] if PROJECT_DIR and PROJECT_DIR not in sys.path else [])]
from src.session import SessionTracker
import src.config as cfg

print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Device: {cfg.Config.DEVICE}")
print(f"Mode: {'Quick test' if QUICK_MODE else 'Full training'}")
print()
for name in ["resnet50", "efficientnet_b0", "vit_b_16"]:
    info = model_results.get(name, {})
    status = info.get("status", "?")
    if status == "ok":
        acc = info.get("accuracy", "?")
        f1 = info.get("weighted_f1", "?")
        print(f"  {name:20s} Acc={acc}  F1={f1}")
    else:
        print(f"  {name:20s} {status}")
if ensemble_metrics:
    print(f"  {'Ensemble':20s} Acc={ensemble_metrics['accuracy']}  F1={ensemble_metrics['weighted_f1']}")

print(f"\n{SessionTracker.summary()}")

from src.visualize import plot_session_comparison
from IPython.display import display, Image

viz_path = plot_session_comparison()
if viz_path:
    display(Image(viz_path))
else:
    print("Not enough sessions to plot comparison.")if "kfold_result" in dir() and kfold_result:
    print(f"  K-Fold CV ({kfold_result[\"n_folds\"]} folds): Acc={kfold_result[\"avg_val_acc\"]:.4f} +/- {kfold_result[\"std_val_acc\"]:.4f}")
if "student_metrics" in dir() and student_metrics:
    print(f"  Distilled {cfg.Config.DISTILL_STUDENT}: Val Acc={student_metrics[\"best_val_acc\"]:.4f}")


In [ ]:
# Push trained models + reports to HuggingFace Hub
import sys, os; PROJECT_DIR = os.getcwd() if os.path.isfile(os.path.join(os.getcwd(), "src", "config.py")) else next((d for d in [os.getcwd(), os.path.join(os.getcwd(), "..")] if os.path.isfile(os.path.join(d, "src", "config.py"))), None); _ = [sys.path.insert(0, p) for p in ([PROJECT_DIR] if PROJECT_DIR and PROJECT_DIR not in sys.path else [])]
from src.utils import sync_models_to_hub
import src.config as cfg

token = input("Enter HF token (write access) or Enter to skip: ").strip()
if token:
    n = sync_models_to_hub(token=token)
    print(f"\nPushed {n} file(s) to {cfg.Config.HF_MODEL_REPO}")
else:
    print("Skipped push.")


In [ ]:
print("=" * 60)
print("Local training complete!")
print("=" * 60)
print("Models: models/")
print("Reports: reports/")
print("Session history: reports/session_results.json")